# Notebook 01 — Ingestion Pipeline Test

Validates the Docling PDF parsing, image extraction, and chunk generation pipeline
on a single sample document from the MMDocRAG corpus.

**What this notebook tests:**
1. Docling layout-aware PDF parsing → structured JSON output
2. Image extraction from parsed layout (figures, charts, tables)
3. Text chunker (512 tokens / 64 overlap)
4. Table chunker (atomic — one chunk per table)
5. Image chunker (one chunk per image with metadata)
6. Chunk count and metadata validation

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import json

## 1. Parse a Sample PDF

In [ ]:
from src.ingestion.docling_parser import DoclingParser

# Pick any PDF from the dataset
sample_pdf = Path('../data/raw/pdfs/COSTCO_2021_10K.pdf')

parser = DoclingParser()
parsed = parser.parse(str(sample_pdf))

print(f'Pages parsed: {len(parsed.get("pages", []))}')
print(f'Layout elements: {len(parsed.get("elements", []))}')
print(f'Keys: {list(parsed.keys())}')

## 2. Extract Images

In [ ]:
from src.ingestion.image_extractor import ImageExtractor

extractor = ImageExtractor(output_dir='../data/raw/images')
images = extractor.extract(str(sample_pdf), doc_name='COSTCO_2021_10K')

print(f'Images extracted: {len(images)}')
if images:
    print('Sample image metadata:', json.dumps(images[0], indent=2))

## 3. Chunk Text

In [ ]:
from src.chunking.text_chunker import TextChunker

chunker = TextChunker(chunk_size=512, chunk_overlap=64)
text_chunks = chunker.chunk(parsed, doc_name='COSTCO_2021_10K')

print(f'Text chunks: {len(text_chunks)}')
if text_chunks:
    c = text_chunks[0]
    print(f'First chunk keys: {list(c.keys())}')
    print(f'First chunk text length: {len(c.get("text", ""))} chars')
    print(f'First chunk page_id: {c.get("page_id")}')

## 4. Chunk Tables

In [ ]:
from src.chunking.table_chunker import TableChunker

table_chunker = TableChunker()
table_chunks = table_chunker.chunk(parsed, doc_name='COSTCO_2021_10K')

print(f'Table chunks: {len(table_chunks)}')
if table_chunks:
    print('Sample table chunk:', json.dumps(table_chunks[0], indent=2)[:500])

## 5. Chunk Images

In [ ]:
from src.chunking.image_chunker import ImageChunker

img_chunker = ImageChunker()
img_chunks = img_chunker.chunk(images, doc_name='COSTCO_2021_10K')

print(f'Image chunks: {len(img_chunks)}')
if img_chunks:
    print('Sample image chunk:', json.dumps(img_chunks[0], indent=2))

## 6. Summary

In [ ]:
all_chunks = text_chunks + table_chunks + img_chunks
print(f'Total chunks for COSTCO_2021_10K:')
print(f'  Text:   {len(text_chunks)}')
print(f'  Tables: {len(table_chunks)}')
print(f'  Images: {len(img_chunks)}')
print(f'  Total:  {len(all_chunks)}')

# Validate required metadata fields
required_fields = {'quote_id', 'doc_name', 'page_id', 'text'}
missing = [c for c in text_chunks if not required_fields.issubset(c.keys())]
print(f'\nText chunks missing required fields: {len(missing)}')